# 🤖 MiniMax Face Refine (Compound Master Hybrid) - Autonomous Batch Runner
### 2-Pass Hybrid: YOLO Face Tracking + SAM3 Semantic Inpainting + VoSR2 Super-Resolution + Gaussian Stitching
* **Mode:** **Autonomous Headless Batch Runner for AI Agents (Antigravity)**
* **Features:** Multi-clip queue, non-blocking fault tolerance, instant output persistence to `/kaggle/working/outputs/`, and auto-termination (`sys.exit(0)`) to preserve GPU quota.


In [ ]:
# [CONFIGURATION] Antigravity Batch Video Restoration Jobs
BATCH_CLIPS = [
    {
        "id": "shot_001_master_refine",
        "picture_1": "picture_1_character_sheet.jpg",
        "picture_2": "picture_2_pottery_wheel_keyframe.jpg",
        "prompt": "high quality sharp natural face, detailed facial micro-texture, cinematic pottery studio, soft studio lighting",
        "steps": 25,
        "cfg": 2.0
    }
]

print(f"Loaded {len(BATCH_CLIPS)} master hybrid restoration jobs for autonomous batch processing.", flush=True)


In [ ]:
# [Step 1/4] Environment & Custom Nodes Setup
import os, sys, subprocess, time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists('ComfyUI'):
    subprocess.run('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git', shell=True, check=True)

custom_nodes = 'ComfyUI/custom_nodes'
os.makedirs(custom_nodes, exist_ok=True)

repos = [
    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'https://github.com/fearnworks/comfyui_fearnworksnodes.git',
    'https://github.com/kijai/ComfyUI-KJNodes.git',
    'https://github.com/drozbay/MaskVidExperiments.git'
]

for repo in repos:
    target = os.path.join(custom_nodes, os.path.splitext(os.path.basename(repo))[0])
    if not os.path.exists(target):
        subprocess.run(f'git clone --depth 1 {repo} {target}', shell=True, check=True)

pkgs = ['huggingface_hub>=0.28.0', 'hf_transfer', 'opencv-python-headless', 'ultralytics', 'imageio', 'imageio-ffmpeg', 'pillow', 'requests', 'tqdm']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ComfyUI/requirements.txt'], check=True)
print('✓ Environment and custom node packages ready.', flush=True)


In [ ]:
# [Step 2/4] High-Speed Model Provisioning (Mounted Dataset + HF Fallback)
import os, shutil, glob, time
from huggingface_hub import hf_hub_download

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

os.makedirs('ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('ComfyUI/models/vae', exist_ok=True)
os.makedirs('ComfyUI/models/latent_upscalers', exist_ok=True)
os.makedirs('ComfyUI/models/upscale_models', exist_ok=True)
os.makedirs('ComfyUI/models/ultralytics/bbox', exist_ok=True)
os.makedirs('ComfyUI/models/checkpoints', exist_ok=True)

if not os.path.exists('ComfyUI/models/clip'):
    try: os.symlink('text_encoders', 'ComfyUI/models/clip')
    except Exception: pass

ds_candidates = glob.glob('/kaggle/input/*minimax*weights*')
ds_root = ds_candidates[0] if ds_candidates else '/kaggle/input/minimax-h3-open-weights'

def link_or_fetch(rel_ds, dst, hf_repo, hf_file):
    src = os.path.join(ds_root, rel_ds)
    if os.path.exists(src):
        if not os.path.exists(dst):
            try: os.symlink(src, dst)
            except Exception: shutil.copyfile(src, dst)
        print(f'✓ [MOUNTED] {dst}', flush=True)
    else:
        if not os.path.exists(dst):
            print(f'⏳ Downloading: {hf_file}...', flush=True)
            p = hf_hub_download(repo_id=hf_repo, filename=hf_file)
            try: os.symlink(p, dst)
            except Exception: shutil.copyfile(p, dst)
            print(f'✓ [DOWNLOADED] {dst}', flush=True)

link_or_fetch('clip/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors')
link_or_fetch('vae/minimax_h3_video_vae_int8_convrot.safetensors', 'ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors', 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors')
link_or_fetch('vae/minimax_h3_audio_vae_fp32.safetensors', 'ComfyUI/models/vae/minimax_h3_audio_vae_fp32.safetensors', 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors')
link_or_fetch('diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', 'ComfyUI/models/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', 'Comfy-Org/MiniMax-H3', 'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors')
link_or_fetch('latent_upscalers/minimax_h3_latent_upscaler_3d_bf16.safetensors', 'ComfyUI/models/latent_upscalers/minimax_h3_latent_upscaler_3d_bf16.safetensors', 'fearnworks/vosr2', 'minimax_h3_latent_upscaler_3d_bf16.safetensors')
link_or_fetch('upscale_models/vosr2_wavelet_fp16.safetensors', 'ComfyUI/models/upscale_models/vosr2_wavelet_fp16.safetensors', 'fearnworks/vosr2', 'vosr2_wavelet_fp16.safetensors')

yolo_dst = 'ComfyUI/models/ultralytics/bbox/face_yolov8m.pt'
if not os.path.exists(yolo_dst):
    subprocess.run(f'wget -q -c https://github.com/akanametov/yolo-face/releases/download/v0.0.0/face_yolov8m.pt -O {yolo_dst}', shell=True)

sam_dst = 'ComfyUI/models/checkpoints/sam3.1_multiplex_fp16.safetensors'
link_or_fetch('checkpoints/sam3.1_multiplex_fp16.safetensors', sam_dst, 'fearnworks/vosr2', 'sam3.1_multiplex_fp16.safetensors')

# Ingest assets into input
input_dir = 'ComfyUI/input'
os.makedirs(input_dir, exist_ok=True)
if os.path.exists('workflows'):
    for f in os.listdir('workflows'):
        if f.lower().endswith(('.jpg', '.png', '.mp4')):
            shutil.copy(os.path.join('workflows', f), input_dir)

print('✓ Checkpoints and assets ready.', flush=True)


In [ ]:
# [Step 3/4] Launch Headless Local ComfyUI Server
import subprocess, time, urllib.request

PORT = 8188
comfy_proc = subprocess.Popen(
    f'{sys.executable} ComfyUI/main.py --listen 127.0.0.1 --port {PORT} --lowvram --reserve-vram 1.5',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

server_ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                print('✓ Headless ComfyUI Server online!', flush=True)
                break
    except Exception:
        time.sleep(2)

if not server_ready:
    raise RuntimeError('ComfyUI Server failed to start.')


In [ ]:
# [Step 4/4] Fault-Tolerant Batch Queue Execution & Auto-Termination
import urllib.request, json, time, shutil, sys

api_wf_file = os.path.join('workflows', 'minimax_h3_r2v_perfect_face_refine_api.json')
with open(api_wf_file, 'r', encoding='utf-8') as f:
    template = json.load(f)

results = []
print('=' * 75, flush=True)
print(f'>>> Executing Master Hybrid Batch Queue: {len(BATCH_CLIPS)} Clips', flush=True)
print('=' * 75, flush=True)

for i, job in enumerate(BATCH_CLIPS, 1):
    job_id = job.get('id', f'clip_{i}')
    print(f'\n[{i}/{len(BATCH_CLIPS)}] Running Master Hybrid Refine: {job_id}...', flush=True)
    t0 = time.time()
    
    try:
        wf = json.loads(json.dumps(template))
        # Override picture inputs if present
        if '9' in wf and 'inputs' in wf['9']:
            wf['9']['inputs']['image'] = job.get('picture_1', 'picture_1_character_sheet.jpg')
        if '257' in wf and 'inputs' in wf['257']:
            wf['257']['inputs']['image'] = job.get('picture_2', 'picture_2_pottery_wheel_keyframe.jpg')
            
        payload = json.dumps({'prompt': wf}).encode('utf-8')
        req = urllib.request.Request(f'http://127.0.0.1:{PORT}/prompt', data=payload, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req) as resp:
            prompt_id = json.loads(resp.read().decode('utf-8'))['prompt_id']

        done = False
        while not done:
            time.sleep(5)
            with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/history/{prompt_id}') as h_resp:
                hist = json.loads(h_resp.read().decode('utf-8'))
                if prompt_id in hist:
                    status = hist[prompt_id].get('status', {})
                    if status.get('completed', False):
                        done = True
                    elif status.get('status_str') == 'error':
                        raise RuntimeError(f"ComfyUI Error: {status.get('messages')}")

        out_files = []
        comfy_out = 'ComfyUI/output'
        if os.path.exists(comfy_out):
            for f in os.listdir(comfy_out):
                if f.endswith('.mp4') or f.endswith('.png'):
                    src_p = os.path.join(comfy_out, f)
                    dst_p = os.path.join(OUTPUT_DIR, f'{job_id}_{f}')
                    shutil.copyfile(src_p, dst_p)
                    out_files.append(dst_p)

        elapsed = time.time() - t0
        print(f'✓ [{job_id}] Rendered in {elapsed:.1f}s -> {out_files}', flush=True)
        results.append({'id': job_id, 'status': 'SUCCESS', 'elapsed': f'{elapsed:.1f}s', 'path': out_files})

    except Exception as e:
        elapsed = time.time() - t0
        print(f'❌ [{job_id}] Error: {e} (Continuing next clip...)', flush=True)
        results.append({'id': job_id, 'status': 'FAILED', 'elapsed': f'{elapsed:.1f}s', 'error': str(e)})

# Save execution summary manifest
with open(os.path.join(OUTPUT_DIR, 'batch_summary.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('\n' + '=' * 75, flush=True)
print('>>> BATCH PROCESSING COMPLETE. SHUTTING DOWN TO PRESERVE GPU QUOTA.', flush=True)
print('=' * 75, flush=True)

comfy_proc.terminate()
sys.exit(0)
